# # 7-03-26 BASELINE MACHINE LEARNING SUBJECT INDIPENDANT

In [1]:
import json
import numpy as np
import torch
from pathlib import Path

project_root = Path("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech")

agg_dir  = project_root / "data/processed/subject_tensors/subject_tensors_aggregated_epoch"
time_dir = project_root / "data/processed/subject_tensors/subject_tensors_time"

agg_files  = sorted(agg_dir.glob("subject_*.pt"))
time_files = sorted(time_dir.glob("subject_*.pt"))

print("agg files :", len(agg_files))
print("time files:", len(time_files))

assert len(agg_files) == len(time_files), "Numero file diverso tra aggregated e time"

X_agg_all = []
X_time_concat_all = []
y_all = []
groups_all = []

for p_agg, p_time in zip(agg_files, time_files):
    subj_agg = torch.load(p_agg, map_location="cpu")
    subj_time = torch.load(p_time, map_location="cpu")

    X_agg = subj_agg["X"].numpy()          # (n_trials, 59, 40)
    y = subj_agg["y"].numpy()

    X_time = subj_time["X"].numpy()        # (n_trials, 5, 59, 40)

    # checks
    assert X_agg.shape[0] == X_time.shape[0], f"n_trials mismatch in {p_agg.name}"
    assert np.all(y == subj_time["y"].numpy()), f"labels mismatch in {p_agg.name}"

    subj_id = int(subj_agg["subject_id"][0].item())

    # aggregated -> mean over channels => (n_trials, 40)
    X_agg_vec = X_agg.mean(axis=1)

    # time-concat -> mean over channels per window, then flatten => (n_trials, 5*40)
    X_time_vec = X_time.mean(axis=2).reshape(X_time.shape[0], -1)

    X_agg_all.append(X_agg_vec)
    X_time_concat_all.append(X_time_vec)
    y_all.append(y)
    groups_all.append(np.full(len(y), subj_id))

X_agg_all = np.concatenate(X_agg_all, axis=0)
X_time_concat_all = np.concatenate(X_time_concat_all, axis=0)
y_all = np.concatenate(y_all, axis=0)
groups_all = np.concatenate(groups_all, axis=0)

print("X_agg_all shape       :", X_agg_all.shape)
print("X_time_concat_all shape:", X_time_concat_all.shape)
print("y_all shape           :", y_all.shape)
print("num subjects          :", len(np.unique(groups_all)))
print("num classes           :", len(np.unique(y_all)))

agg files : 74
time files: 74
X_agg_all shape       : (38926, 40)
X_time_concat_all shape: (38926, 200)
y_all shape           : (38926,)
num subjects          : 74
num classes           : 110


In [2]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

def eval_subject_independent(X, y, groups, random_state=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    g_train = groups[train_idx]

    # ulteriore split train/val sempre per gruppi
    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    tr_idx_local, val_idx_local = next(gss_val.split(X_train, y_train, groups=g_train))

    X_tr, X_val = X_train[tr_idx_local], X_train[val_idx_local]
    y_tr, y_val = y_train[tr_idx_local], y_train[val_idx_local]

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=3000,
            n_jobs=-1,
            multi_class="multinomial"
        )
    )

    clf.fit(X_tr, y_tr)

    y_val_pred = clf.predict(X_val)
    y_test_pred = clf.predict(X_test)

    return {
        "val_acc": accuracy_score(y_val, y_val_pred),
        "val_bal_acc": balanced_accuracy_score(y_val, y_val_pred),
        "test_acc": accuracy_score(y_test, y_test_pred),
        "test_bal_acc": balanced_accuracy_score(y_test, y_test_pred),
        "n_train": len(X_tr),
        "n_val": len(X_val),
        "n_test": len(X_test),
        "n_subjects_train": len(np.unique(groups[train_idx][tr_idx_local])),
        "n_subjects_val": len(np.unique(groups[train_idx][val_idx_local])),
        "n_subjects_test": len(np.unique(groups[test_idx])),
    }

In [3]:
res_agg = eval_subject_independent(X_agg_all, y_all, groups_all, random_state=42)
res_time = eval_subject_independent(X_time_concat_all, y_all, groups_all, random_state=42)

print("=== AGGREGATED ===")
for k, v in res_agg.items():
    print(f"{k}: {v}")

print("\n=== TIME-CONCAT ===")
for k, v in res_time.items():
    print(f"{k}: {v}")

/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


=== AGGREGATED ===
val_acc: 0.008517110266159696
val_bal_acc: 0.008509237007957918
test_acc: 0.00875796178343949
test_bal_acc: 0.00875765325893203
n_train: 24815
n_val: 6575
n_test: 7536
n_subjects_train: 47
n_subjects_val: 12
n_subjects_test: 15

=== TIME-CONCAT ===
val_acc: 0.011406844106463879
val_bal_acc: 0.011393422299181562
test_acc: 0.009554140127388535
test_bal_acc: 0.009571528216029494
n_train: 24815
n_val: 6575
n_test: 7536
n_subjects_train: 47
n_subjects_val: 12
n_subjects_test: 15


In [4]:
seeds = [0, 1, 2, 3, 4]

agg_accs = []
time_accs = []
agg_baccs = []
time_baccs = []

for seed in seeds:
    r_agg = eval_subject_independent(X_agg_all, y_all, groups_all, random_state=seed)
    r_time = eval_subject_independent(X_time_concat_all, y_all, groups_all, random_state=seed)

    agg_accs.append(r_agg["test_acc"])
    time_accs.append(r_time["test_acc"])
    agg_baccs.append(r_agg["test_bal_acc"])
    time_baccs.append(r_time["test_bal_acc"])

print("AGG test acc       :", np.mean(agg_accs), "±", np.std(agg_accs))
print("TIME test acc      :", np.mean(time_accs), "±", np.std(time_accs))
print("AGG test bal_acc   :", np.mean(agg_baccs), "±", np.std(agg_baccs))
print("TIME test bal_acc  :", np.mean(time_baccs), "±", np.std(time_baccs))

/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/homebrew

AGG test acc       : 0.008861524583805415 ± 0.0012128055424852492
TIME test acc      : 0.009420803455924992 ± 0.0003976406202506901
AGG test bal_acc   : 0.00885313217844151 ± 0.0012071952981626876
TIME test bal_acc  : 0.00942131681009031 ± 0.0003989123885634392


## BASELINE MLP 

In [5]:
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score

In [6]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

def fit_scaler_and_transform(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)
    return X_train_s, X_val_s, X_test_s, scaler


def train_mlp(
    X_train, y_train,
    X_val, y_val,
    n_classes,
    hidden_dim=256,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=100,
    patience=10,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.long)

    train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = MLP(in_dim=X_train.shape[1], hidden_dim=hidden_dim, out_dim=n_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    best_state = None
    best_val_bacc = -1.0
    wait = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t.to(device)).cpu().numpy()
        val_pred = val_logits.argmax(axis=1)
        val_bacc = balanced_accuracy_score(y_val, val_pred)

        if val_bacc > best_val_bacc:
            best_val_bacc = val_bacc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model


def eval_model(model, X, y, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        logits = model(X_t).cpu().numpy()

    pred = logits.argmax(axis=1)
    return {
        "acc": accuracy_score(y, pred),
        "bal_acc": balanced_accuracy_score(y, pred),
        "pred": pred,
    }

In [7]:
project_root = Path("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech")

agg_dir  = project_root / "data/processed/subject_tensors/subject_tensors_aggregated_epoch"
time_dir = project_root / "data/processed/subject_tensors/subject_tensors_time"

agg_files  = sorted(agg_dir.glob("subject_*.pt"))
time_files = sorted(time_dir.glob("subject_*.pt"))

assert len(agg_files) == len(time_files), "Numero file diverso tra agg e time"

X_agg_all = []
X_time_all = []
y_all = []
groups_all = []

for p_agg, p_time in zip(agg_files, time_files):
    d_agg = torch.load(p_agg, map_location="cpu")
    d_time = torch.load(p_time, map_location="cpu")

    X_agg = d_agg["X"].numpy()        # (E, 59, 40)
    y = d_agg["y"].numpy()

    X_time = d_time["X"].numpy()      # (E, 5, 59, 40)

    assert X_agg.shape[0] == X_time.shape[0], f"Mismatch {p_agg.name}"
    assert np.all(y == d_time["y"].numpy()), f"Label mismatch {p_agg.name}"

    subj_id = int(d_agg["subject_id"][0].item())

    # aggregated -> mean sui canali => (E, 40)
    X_agg_vec = X_agg.mean(axis=1)

    # time -> mean sui canali per window => (E, 5, 40) -> flatten => (E, 200)
    X_time_vec = X_time.mean(axis=2).reshape(X_time.shape[0], -1)

    X_agg_all.append(X_agg_vec)
    X_time_all.append(X_time_vec)
    y_all.append(y)
    groups_all.append(np.full(len(y), subj_id))

X_agg_all = np.concatenate(X_agg_all, axis=0)
X_time_all = np.concatenate(X_time_all, axis=0)
y_all = np.concatenate(y_all, axis=0)
groups_all = np.concatenate(groups_all, axis=0)

print("X_agg_all :", X_agg_all.shape)
print("X_time_all:", X_time_all.shape)
print("y_all     :", y_all.shape)
print("subjects  :", len(np.unique(groups_all)))
print("classes   :", len(np.unique(y_all)))

X_agg_all : (38926, 40)
X_time_all: (38926, 200)
y_all     : (38926,)
subjects  : 74
classes   : 110


## MLP SUBJECT SPECIFIC

In [9]:
from collections import defaultdict

SUBJECT_ID = 10

mask = groups_all == SUBJECT_ID

X_agg_sub = X_agg_all[mask]
X_time_sub = X_time_all[mask]
y_sub = y_all[mask]

print("Subject:", SUBJECT_ID)
print("Samples :", len(y_sub))
print("Classes :", len(np.unique(y_sub)))

# ------------------------------------------------------------
# split fisso per classe: 3 train / 1 val / 1 test
# ------------------------------------------------------------
rng = np.random.RandomState(42)

class_to_indices = defaultdict(list)
for i, yy in enumerate(y_sub):
    class_to_indices[int(yy)].append(i)

idx_train = []
idx_val = []
idx_test = []

for cls, idxs in class_to_indices.items():
    idxs = np.array(idxs)
    rng.shuffle(idxs)

    # ci aspettiamo 5 trial per classe
    if len(idxs) < 5:
        print(f"⚠ class {cls} has only {len(idxs)} samples, skipping")
        continue

    idx_train.extend(idxs[:3])
    idx_val.extend(idxs[3:4])
    idx_test.extend(idxs[4:5])

idx_train = np.array(idx_train)
idx_val = np.array(idx_val)
idx_test = np.array(idx_test)

print("train:", len(idx_train))
print("val  :", len(idx_val))
print("test :", len(idx_test))

Subject: 10
Samples : 550
Classes : 110
train: 330
val  : 110
test : 110


In [10]:
def run_subject_specific(X, y, name):
    X_train, X_val, X_test = X[idx_train], X[idx_val], X[idx_test]
    y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

    X_train, X_val, X_test, scaler = fit_scaler_and_transform(X_train, X_val, X_test)

    model = train_mlp(
        X_train, y_train,
        X_val, y_val,
        n_classes=len(np.unique(y_all)),   # 110 output globali
        hidden_dim=256,
        lr=1e-3,
        batch_size=64,
        max_epochs=100,
        patience=10,
    )

    val_res = eval_model(model, X_val, y_val)
    test_res = eval_model(model, X_test, y_test)

    print(f"\n=== {name} | subject-specific | subject {SUBJECT_ID} ===")
    print("val acc     :", val_res["acc"])
    print("val bal_acc :", val_res["bal_acc"])
    print("test acc    :", test_res["acc"])
    print("test bal_acc:", test_res["bal_acc"])

In [11]:
run_subject_specific(X_agg_sub, y_sub, "AGG")
run_subject_specific(X_time_sub, y_sub, "TIME")


=== AGG | subject-specific | subject 10 ===
val acc     : 0.00909090909090909
val bal_acc : 0.00909090909090909
test acc    : 0.00909090909090909
test bal_acc: 0.00909090909090909

=== TIME | subject-specific | subject 10 ===
val acc     : 0.01818181818181818
val bal_acc : 0.01818181818181818
test acc    : 0.00909090909090909
test bal_acc: 0.00909090909090909


## MLP SUBJECT INDIPENDANT

In [12]:
def run_subject_independent(X, y, groups, name, seed=42):
    gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss_test.split(X, y, groups))

    X_train_full, X_test = X[train_idx], X[test_idx]
    y_train_full, y_test = y[train_idx], y[test_idx]
    g_train_full = groups[train_idx]

    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_local, val_local = next(gss_val.split(X_train_full, y_train_full, g_train_full))

    X_train = X_train_full[tr_local]
    y_train = y_train_full[tr_local]

    X_val = X_train_full[val_local]
    y_val = y_train_full[val_local]

    X_train, X_val, X_test, scaler = fit_scaler_and_transform(X_train, X_val, X_test)

    model = train_mlp(
        X_train, y_train,
        X_val, y_val,
        n_classes=len(np.unique(y_all)),
        hidden_dim=256,
        lr=1e-3,
        batch_size=128,
        max_epochs=100,
        patience=10,
    )

    val_res = eval_model(model, X_val, y_val)
    test_res = eval_model(model, X_test, y_test)

    print(f"\n=== {name} | subject-independent ===")
    print("train subjects:", len(np.unique(g_train_full[tr_local])))
    print("val subjects  :", len(np.unique(g_train_full[val_local])))
    print("test subjects :", len(np.unique(groups[test_idx])))
    print("val acc       :", val_res["acc"])
    print("val bal_acc   :", val_res["bal_acc"])
    print("test acc      :", test_res["acc"])
    print("test bal_acc  :", test_res["bal_acc"])

run_subject_independent(X_agg_all, y_all, groups_all, "AGG", seed=42)
run_subject_independent(X_time_all, y_all, groups_all, "TIME", seed=42)


=== AGG | subject-independent ===
train subjects: 47
val subjects  : 12
test subjects : 15
val acc       : 0.00988593155893536
val bal_acc   : 0.009872439313950846
test acc      : 0.009156050955414012
test bal_acc  : 0.009118255998051394

=== TIME | subject-independent ===
train subjects: 47
val subjects  : 12
test subjects : 15
val acc       : 0.01064638783269962
val bal_acc   : 0.010638414595021314
test acc      : 0.009288747346072187
test bal_acc  : 0.009300218110959798


In [13]:
seeds = [0, 1, 2, 3, 4]

agg_accs, agg_baccs = [], []
time_accs, time_baccs = [], []

for seed in seeds:
    print(f"\n######## seed = {seed} ########")

    def get_metrics(X, y, groups):
        gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        train_idx, test_idx = next(gss_test.split(X, y, groups))

        X_train_full, X_test = X[train_idx], X[test_idx]
        y_train_full, y_test = y[train_idx], y[test_idx]
        g_train_full = groups[train_idx]

        gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        tr_local, val_local = next(gss_val.split(X_train_full, y_train_full, g_train_full))

        X_train = X_train_full[tr_local]
        y_train = y_train_full[tr_local]
        X_val = X_train_full[val_local]
        y_val = y_train_full[val_local]

        X_train, X_val, X_test, _ = fit_scaler_and_transform(X_train, X_val, X_test)

        model = train_mlp(
            X_train, y_train,
            X_val, y_val,
            n_classes=len(np.unique(y_all)),
            hidden_dim=256,
            lr=1e-3,
            batch_size=128,
            max_epochs=100,
            patience=10,
        )

        test_res = eval_model(model, X_test, y_test)
        return test_res["acc"], test_res["bal_acc"]

    a_acc, a_bacc = get_metrics(X_agg_all, y_all, groups_all)
    t_acc, t_bacc = get_metrics(X_time_all, y_all, groups_all)

    agg_accs.append(a_acc)
    agg_baccs.append(a_bacc)
    time_accs.append(t_acc)
    time_baccs.append(t_bacc)

print("\n=== SUBJECT-INDEPENDENT SUMMARY ===")
print("AGG  acc     :", np.mean(agg_accs), "±", np.std(agg_accs))
print("AGG  bal_acc :", np.mean(agg_baccs), "±", np.std(agg_baccs))
print("TIME acc     :", np.mean(time_accs), "±", np.std(time_accs))
print("TIME bal_acc :", np.mean(time_baccs), "±", np.std(time_baccs))


######## seed = 0 ########

######## seed = 1 ########

######## seed = 2 ########

######## seed = 3 ########

######## seed = 4 ########

=== SUBJECT-INDEPENDENT SUMMARY ===
AGG  acc     : 0.00896704722868604 ± 0.0005099379326317533
AGG  bal_acc : 0.008950928816588289 ± 0.0005125172410760806
TIME acc     : 0.009058852301777085 ± 0.0005965683412382097
TIME bal_acc : 0.00906245039678757 ± 0.0006033985707244703


# FULL FLATTEN MLP TEST

In [14]:
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from collections import defaultdict

In [15]:
project_root = Path("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech")

agg_dir  = project_root / "data/processed/subject_tensors/subject_tensors_aggregated_epoch"
time_dir = project_root / "data/processed/subject_tensors/subject_tensors_time"

agg_files  = sorted(agg_dir.glob("subject_*.pt"))
time_files = sorted(time_dir.glob("subject_*.pt"))

assert len(agg_files) == len(time_files), "Numero file diverso tra aggregated e time"

X_agg_all = []
X_time_all = []
y_all = []
groups_all = []

for p_agg, p_time in zip(agg_files, time_files):
    d_agg = torch.load(p_agg, map_location="cpu")
    d_time = torch.load(p_time, map_location="cpu")

    X_agg = d_agg["X"].numpy()      # (E, 59, 40)
    y = d_agg["y"].numpy()

    X_time = d_time["X"].numpy()    # (E, 5, 59, 40)

    assert X_agg.shape[0] == X_time.shape[0], f"Mismatch trials in {p_agg.name}"
    assert np.all(y == d_time["y"].numpy()), f"Mismatch labels in {p_agg.name}"

    subj_id = int(d_agg["subject_id"][0].item())

    # FULL FLATTEN
    X_agg_vec = X_agg.reshape(X_agg.shape[0], -1)        # (E, 2360)
    X_time_vec = X_time.reshape(X_time.shape[0], -1)     # (E, 11800)

    X_agg_all.append(X_agg_vec)
    X_time_all.append(X_time_vec)
    y_all.append(y)
    groups_all.append(np.full(len(y), subj_id))

X_agg_all = np.concatenate(X_agg_all, axis=0)
X_time_all = np.concatenate(X_time_all, axis=0)
y_all = np.concatenate(y_all, axis=0)
groups_all = np.concatenate(groups_all, axis=0)

print("X_agg_all :", X_agg_all.shape)
print("X_time_all:", X_time_all.shape)
print("y_all     :", y_all.shape)
print("subjects  :", len(np.unique(groups_all)))
print("classes   :", len(np.unique(y_all)))

X_agg_all : (38926, 2360)
X_time_all: (38926, 11800)
y_all     : (38926,)
subjects  : 74
classes   : 110


In [16]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

def fit_scaler_and_transform(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)
    return X_train_s, X_val_s, X_test_s, scaler


def train_mlp(
    X_train, y_train,
    X_val, y_val,
    n_classes,
    hidden_dim=512,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=100,
    patience=10,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.long)

    train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = MLP(in_dim=X_train.shape[1], hidden_dim=hidden_dim, out_dim=n_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    best_state = None
    best_val_bacc = -1.0
    wait = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t.to(device)).cpu().numpy()

        val_pred = val_logits.argmax(axis=1)
        val_bacc = balanced_accuracy_score(y_val, val_pred)

        if val_bacc > best_val_bacc:
            best_val_bacc = val_bacc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model


def eval_model(model, X, y, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    model = model.to(device)
    model.eval()

    with torch.no_grad():
        logits = model(X_t).cpu().numpy()

    pred = logits.argmax(axis=1)

    return {
        "acc": accuracy_score(y, pred),
        "bal_acc": balanced_accuracy_score(y, pred),
        "pred": pred,
    }

 ## Test 1 — subject-specific full flatten

In [17]:
SUBJECT_ID = 10

mask = groups_all == SUBJECT_ID

X_agg_sub = X_agg_all[mask]
X_time_sub = X_time_all[mask]
y_sub = y_all[mask]

print("Subject:", SUBJECT_ID)
print("Samples :", len(y_sub))
print("Classes :", len(np.unique(y_sub)))
print("X_agg_sub :", X_agg_sub.shape)
print("X_time_sub:", X_time_sub.shape)

Subject: 10
Samples : 550
Classes : 110
X_agg_sub : (550, 2360)
X_time_sub: (550, 11800)


In [18]:
rng = np.random.RandomState(42)

class_to_indices = defaultdict(list)
for i, yy in enumerate(y_sub):
    class_to_indices[int(yy)].append(i)

idx_train = []
idx_val = []
idx_test = []

for cls, idxs in class_to_indices.items():
    idxs = np.array(idxs)
    rng.shuffle(idxs)

    if len(idxs) < 5:
        print(f"⚠ class {cls} has only {len(idxs)} samples, skipping")
        continue

    idx_train.extend(idxs[:3])
    idx_val.extend(idxs[3:4])
    idx_test.extend(idxs[4:5])

idx_train = np.array(idx_train)
idx_val = np.array(idx_val)
idx_test = np.array(idx_test)

print("train:", len(idx_train))
print("val  :", len(idx_val))
print("test :", len(idx_test))

def run_subject_specific_full(X, y, name):
    X_train, X_val, X_test = X[idx_train], X[idx_val], X[idx_test]
    y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

    X_train, X_val, X_test, _ = fit_scaler_and_transform(X_train, X_val, X_test)

    model = train_mlp(
        X_train, y_train,
        X_val, y_val,
        n_classes=len(np.unique(y_all)),
        hidden_dim=512,
        lr=1e-3,
        batch_size=64,
        max_epochs=100,
        patience=10,
    )

    val_res = eval_model(model, X_val, y_val)
    test_res = eval_model(model, X_test, y_test)

    print(f"\n=== {name} | subject-specific FULL | subject {SUBJECT_ID} ===")
    print("val acc     :", val_res["acc"])
    print("val bal_acc :", val_res["bal_acc"])
    print("test acc    :", test_res["acc"])
    print("test bal_acc:", test_res["bal_acc"])

train: 330
val  : 110
test : 110


In [19]:
run_subject_specific_full(X_agg_sub, y_sub, "AGG")
run_subject_specific_full(X_time_sub, y_sub, "TIME")

/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)



=== AGG | subject-specific FULL | subject 10 ===
val acc     : 0.01818181818181818
val bal_acc : 0.01818181818181818
test acc    : 0.0
test bal_acc: 0.0


/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
/opt/homebrew/Caskroom/miniconda/base/envs/daniele_310/lib/python3.10/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_p


=== TIME | subject-specific FULL | subject 10 ===
val acc     : 0.00909090909090909
val bal_acc : 0.00909090909090909
test acc    : 0.00909090909090909
test bal_acc: 0.00909090909090909


## Test 2 — subject-independent full flatten

In [20]:
def run_subject_independent_full(X, y, groups, name, seed=42):
    gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss_test.split(X, y, groups))

    X_train_full, X_test = X[train_idx], X[test_idx]
    y_train_full, y_test = y[train_idx], y[test_idx]
    g_train_full = groups[train_idx]

    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_local, val_local = next(gss_val.split(X_train_full, y_train_full, g_train_full))

    X_train = X_train_full[tr_local]
    y_train = y_train_full[tr_local]
    X_val = X_train_full[val_local]
    y_val = y_train_full[val_local]

    X_train, X_val, X_test, _ = fit_scaler_and_transform(X_train, X_val, X_test)

    model = train_mlp(
        X_train, y_train,
        X_val, y_val,
        n_classes=len(np.unique(y_all)),
        hidden_dim=512,
        lr=1e-3,
        batch_size=128,
        max_epochs=100,
        patience=10,
    )

    val_res = eval_model(model, X_val, y_val)
    test_res = eval_model(model, X_test, y_test)

    print(f"\n=== {name} | subject-independent FULL ===")
    print("train subjects:", len(np.unique(g_train_full[tr_local])))
    print("val subjects  :", len(np.unique(g_train_full[val_local])))
    print("test subjects :", len(np.unique(groups[test_idx])))
    print("val acc       :", val_res["acc"])
    print("val bal_acc   :", val_res["bal_acc"])
    print("test acc      :", test_res["acc"])
    print("test bal_acc  :", test_res["bal_acc"])

In [21]:
run_subject_independent_full(X_agg_all, y_all, groups_all, "AGG", seed=42)
run_subject_independent_full(X_time_all, y_all, groups_all, "TIME", seed=42)


=== AGG | subject-independent FULL ===
train subjects: 47
val subjects  : 12
test subjects : 15
val acc       : 0.00988593155893536
val bal_acc   : 0.009866983833602912
test acc      : 0.00875796178343949
test bal_acc  : 0.008577130456925854

=== TIME | subject-independent FULL ===
train subjects: 47
val subjects  : 12
test subjects : 15
val acc       : 0.009581749049429657
val bal_acc   : 0.009602035919068427
test acc      : 0.009023354564755838
test bal_acc  : 0.00909090909090909


In [ ]:
seeds = [0, 1, 2, 3, 4]

agg_accs, agg_baccs = [], []
time_accs, time_baccs = [], []

for seed in seeds:
    def get_metrics_full(X, y, groups):
        gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        train_idx, test_idx = next(gss_test.split(X, y, groups))

        X_train_full, X_test = X[train_idx], X[test_idx]
        y_train_full, y_test = y[train_idx], y[test_idx]
        g_train_full = groups[train_idx]

        gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        tr_local, val_local = next(gss_val.split(X_train_full, y_train_full, g_train_full))

        X_train = X_train_full[tr_local]
        y_train = y_train_full[tr_local]
        X_val = X_train_full[val_local]
        y_val = y_train_full[val_local]

        X_train, X_val, X_test, _ = fit_scaler_and_transform(X_train, X_val, X_test)

        model = train_mlp(
            X_train, y_train,
            X_val, y_val,
            n_classes=len(np.unique(y_all)),
            hidden_dim=512,
            lr=1e-3,
            batch_size=128,
            max_epochs=100,
            patience=10,
        )

        test_res = eval_model(model, X_test, y_test)
        return test_res["acc"], test_res["bal_acc"]

    a_acc, a_bacc = get_metrics_full(X_agg_all, y_all, groups_all)
    t_acc, t_bacc = get_metrics_full(X_time_all, y_all, groups_all)

    agg_accs.append(a_acc)
    agg_baccs.append(a_bacc)
    time_accs.append(t_acc)
    time_baccs.append(t_bacc)

print("\n=== SUBJECT-INDEPENDENT FULL SUMMARY ===")
print("AGG  acc     :", np.mean(agg_accs), "±", np.std(agg_accs))
print("AGG  bal_acc :", np.mean(agg_baccs), "±", np.std(agg_baccs))
print("TIME acc     :", np.mean(time_accs), "±", np.std(time_accs))
print("TIME bal_acc :", np.mean(time_baccs), "±", np.std(time_baccs))